# 02 - Data Extraction

This notebook tests the AI-powered invoice data extraction using Claude.


In [1]:
from src.document_processor import DocumentProcessor
from src.ai_extractor import AIExtractor
from src.config import config
import json


In [2]:
# Initialize processors
doc_processor = DocumentProcessor(config.directories.invoices)
ai_extractor = AIExtractor(config)

print(f"AI Model: {config.ai.model}")
print(f"Invoices directory: {config.directories.invoices}")


AI Model: claude-3-5-sonnet-20241022
Invoices directory: /Users/pavelzverina/AiProjects/fakturoid/data/invoices


In [3]:
# Get list of invoice files
files = doc_processor.list_invoice_files()
print(f"Found {len(files)} invoice files:")
for f in files:
    print(f"  - {f.name}")


Found 4 invoice files:
  - Alien Isolation.pdf
  - FP20250158 - OrderSummary202509013059566415002039.png
  - OpenAI-Invoice-3D6B9186-0031.pdf
  - google-workspace-5369924648.pdf


In [4]:
# Test extraction on first file
if files:
    test_file = files[0]
    print(f"Testing extraction on: {test_file.name}\n")
    
    # Extract data
    invoice_data = ai_extractor.extract_invoice_data(test_file)
    
    # Display results
    print("Extracted Invoice Data:")
    print(json.dumps(invoice_data, indent=2, ensure_ascii=False))
else:
    print("No invoice files found.")


Testing extraction on: Alien Isolation.pdf

Extracted Invoice Data:
{
  "invoice_number": "786940972572357",
  "issue_date": "2025-10-02",
  "supplier_name": "Sony Interactive Entertainment Network Europe Limited",
  "total_amount": 207.25,
  "due_date": null,
  "variable_symbol": null,
  "supplier_address": "10 Great Marlborough Street, London, W1F 7LP",
  "supplier_ico": null,
  "supplier_dic": null,
  "currency": "Kc",
  "tax_amount": 35.97,
  "line_items": [
    {
      "description": "Alien: Isolation (Game)",
      "total": 207.25
    }
  ],
  "notes": "This is not a VAT/GST invoice. This email message has been delivered from a send-only address.",
  "confidence": null,
  "source_file": "Alien Isolation.pdf"
}


In [5]:
# Test extraction on all files
all_results = []

for invoice_file in files:
    print(f"\nProcessing: {invoice_file.name}")
    try:
        data = ai_extractor.extract_invoice_data(invoice_file)
        all_results.append({
            "filename": invoice_file.name,
            "data": data,
            "status": "success"
        })
        print(f"  ✓ Invoice: {data.get('invoice_number', 'N/A')}")
        print(f"  ✓ Amount: {data.get('total_amount', 'N/A')} {data.get('currency', 'CZK')}")
    except Exception as e:
        all_results.append({
            "filename": invoice_file.name,
            "error": str(e),
            "status": "failed"
        })
        print(f"  ✗ Error: {e}")

print(f"\n\nSummary: {len([r for r in all_results if r['status'] == 'success'])}/{len(files)} successful")



Processing: Alien Isolation.pdf
  ✓ Invoice: 786940972572357
  ✓ Amount: 207.25 Kc

Processing: FP20250158 - OrderSummary202509013059566415002039.png
  ✓ Invoice: 3059566415002039
  ✓ Amount: 105.07 CZK

Processing: OpenAI-Invoice-3D6B9186-0031.pdf
  ✓ Invoice: 3D6B9186-0031
  ✓ Amount: 20.0 USD

Processing: google-workspace-5369924648.pdf
  ✓ Invoice: 5369924648
  ✓ Amount: 8.1 EUR


Summary: 4/4 successful


In [ ]:
# Display AI usage summary
ai_extractor.print_usage_summary()
